# Search three classifiers, then choose one

## Objectives

- Search small grids for logistic regression, a random forest, and an SVM.
- Compare the three on validation ROC-AUC and pick one winner.
- Keep the test rows closed until the model evaluation part.

## What we will do

1. Fit the logistic and forest searches on the train rows with the train and cross-validation pipeline: engineered columns, scaling, one-hot encoding, and `SelectKBest` with 8 features.
2. Build the SVM candidate with `CalibratedClassifierCV` around `SVC`, then search `C`.
3. Print the validation comparison table and set `chosen_model` to the highest validation ROC-AUC.
4. Write one sentence about why you kept that model, then continue to the model evaluation part.

`SVC(probability=True)` is deprecated in scikit-learn 1.9 and removed in 1.11. `CalibratedClassifierCV(SVC(kernel="rbf", random_state=42), ensemble=False)` still returns probabilities for the threshold and the form.

If a search is still running after a minute, stop it and set `n_jobs=-1`, or keep only the first value in each list. On the machine used to write these ranges, each search finished in under 2 seconds.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "app.py").is_file() and (candidate / "lib" / "schema.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "app.py").is_file() and (nested / "lib" / "schema.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder or from 01-supervised-learning."
    )
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from pipelines import fit_named

logistic_search, logistic_seconds = fit_named("logistic", X_train, y_train, "churn")
forest_search, forest_seconds = fit_named("forest", X_train, y_train, "churn")
print("logistic and forest searches finished")
print("logistic seconds", round(logistic_seconds, 1))
print("forest seconds", round(forest_seconds, 1))


## Add the SVM

Build `svc_pipeline` with `make_candidate_pipeline`. The model step is `CalibratedClassifierCV(SVC(kernel="rbf", random_state=42), ensemble=False)`.

`svc_grid` searches `model__estimator__C` at `1.0` and `10.0`.

Steps:

1. Build the calibrated SVC estimator: `CalibratedClassifierCV(SVC(kernel="rbf", random_state=42), ensemble=False)`.
2. Pass that estimator to `make_candidate_pipeline` and store the result in `svc_pipeline`.
3. Set `svc_grid` to a dict that searches `model__estimator__C` over `[1.0, 10.0]`.

The next cell fits that grid on the train rows and prints the comparison. The majority baseline from the split part has ROC-AUC 0.5 and is not a candidate. Expect logistic validation ROC-AUC near 0.80, the forest a little lower, and the SVM near 0.75. Each candidate should beat 0.5. Pick the highest validation ROC-AUC. Do not use the test rows.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

from pipelines import make_candidate_pipeline

svc_pipeline = None
svc_grid = None


In [ ]:
import time

from sklearn.model_selection import GridSearchCV

from pipelines import churn_folds, format_comparison, score_row

if svc_pipeline is None or svc_grid is None:
    raise NotImplementedError("Build the SVC pipeline and its C grid.")
started = time.perf_counter()
svc_search = GridSearchCV(
    svc_pipeline,
    svc_grid,
    scoring="roc_auc",
    cv=churn_folds(),
    refit=True,
    n_jobs=1,
)
svc_search.fit(X_train, y_train)
svc_seconds = time.perf_counter() - started
rows = [
    score_row("logistic", logistic_search, logistic_seconds, X_val, y_val, "churn"),
    score_row("forest", forest_search, forest_seconds, X_val, y_val, "churn"),
    score_row("svc", svc_search, svc_seconds, X_val, y_val, "churn"),
]
print(format_comparison(rows, "churn"))
print("majority baseline val_auc 0.500 (not a candidate; from the split part)")


### Exercise

`chosen_model` is `"logistic"`, `"forest"`, or `"svc"`. Set it to the name with the highest validation ROC-AUC in the table. Expect `logistic`.

Steps:

1. Read the validation ROC-AUC column in the comparison table above.
2. Pick the name with the highest validation ROC-AUC.
3. Assign that string to `chosen_model`.
4. Under the code cell, write one sentence about why you kept that model.

Do not look at the test rows here. The model evaluation part uses the test set once.


In [ ]:
chosen_model = None


In [ ]:
if chosen_model not in {"logistic", "forest", "svc"}:
    raise NotImplementedError("Set chosen_model to logistic, forest, or svc.")
print("chosen", chosen_model)
